## Loading all necessary silver datasets

In [0]:
customers_df = spark.table("project.silver.silver_customers")
orders_df = spark.table("project.silver.silver_orders")
order_items_df = spark.table("project.silver.silver_items")
products_df = spark.table("project.silver.silver_product")
payments_df = spark.table("project.silver.silver_payments")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

In [0]:
delivered_orders = orders_df.filter(F.col("order_status") == "delivered") \
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_estimated_delivery_date", F.to_date("order_estimated_delivery_date")) \
    .withColumn("order_delivered_customer_date", F.to_date("order_delivered_customer_date")) \
    .dropna(subset=["order_delivered_customer_date"]) # Remove rows with missing actual delivery dates


In [0]:
master_df = delivered_orders \
    .join(customers_df.select("customer_id", "customer_state"), on="customer_id") \
    .join(order_items_df.select("order_id", "price", "freight_value", "product_id"), on="order_id") \
    .join(products_df.select("product_id", "product_category_name_english"), on="product_id") \
    .join(payments_df.groupBy("order_id").agg(F.max("payment_installments").alias("installments")), on="order_id", how="left")


In [0]:
# 4. Feature Engineering
gold_table = master_df.withColumn(
    # Time-based features
    "order_dow", F.dayofweek("order_purchase_timestamp")
).withColumn(
    "order_hour", F.hour("order_purchase_timestamp")
).withColumn(
    "order_month", F.month("order_purchase_timestamp")
).withColumn(
    # Promised lead time (How many days did Olist tell the customer it would take?)
    "estimated_days", F.datediff("order_estimated_delivery_date", "order_purchase_timestamp")
).withColumn(
    # THE TARGET VARIABLE: is_late
    # 1 if delivered after estimated date, 0 if on-time or early
    "is_late", F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1).otherwise(0)
)

In [0]:
# 5. Final Selection of ML-Ready Columns
final_gold_columns = [
    "order_id", 
    "customer_state", 
    "product_category_name_english",
    "price", 
    "freight_value", 
    "installments",
    "order_dow", 
    "order_hour", 
    "order_month",
    "estimated_days", 
    "is_late" # Target
]


In [0]:
delivery_delay_ml_data = gold_table.select(*final_gold_columns).fillna({"installments": 1, "product_category_name_english": "unknown"})

In [0]:
# 6. Display to check for variance and class balance
print(f"Total Rows for ML: {delivery_delay_ml_data.count()}")
display(delivery_delay_ml_data.groupBy("is_late").count())


Total Rows for ML: 106225


is_late,count
0,99162
1,7063


In [0]:
delivery_delay_ml_data.toPandas().to_csv("delivery_delay.csv")

In [0]:
display(delivery_delay_ml_data.head(10))

order_id,customer_state,product_category_name_english,price,freight_value,installments,order_dow,order_hour,order_month,estimated_days,is_late
0010b2e5201cc5f1ae7e9c6cc8f5bd00,RJ,cool_stuff,48.9,16.6,3,2,17,9,16,0
0028de0ca693a1bb26448916a81105cc,RS,construction_tools_lights,29.99,15.31,1,4,14,8,27,0
0032d07457ae9c806c79368d7d9ce96b,RJ,housewares,159.0,27.19,2,7,18,3,38,1
00471463a6106056c1a2a809f70de640,DF,furniture_decor,179.99,85.97,5,5,22,9,29,0
0068e836900867da359bd81db9227a33,RS,sports_leisure,96.99,16.44,1,6,11,9,25,0
006f7dfffe2d90809598e8f1972b829b,SP,health_beauty,39.85,12.79,3,5,22,3,20,0
008819c6d5f6da6fa5cd9d50ca927cf4,SP,watches_gifts,55.0,7.65,2,7,21,8,5,0
00891ba5de66f55000ee358ceea9b345,DF,sports_leisure,133.0,16.69,4,1,17,5,24,0
00949655cdd8d0465e433a5e6c9643e5,SP,construction_tools_construction,18.0,7.87,1,2,21,6,10,0
0095790a64527ec83aeaaf99023c050e,MA,watches_gifts,19.9,42.76,6,4,13,11,26,0


In [0]:
display(delivery_delay_ml_data.groupBy("product_category_name_english").count())

product_category_name_english,count
health_beauty,9183
fashion_underwear_beach,118
sports_leisure,8081
electronics,2685
perfumery,3249
air_conditioning,275
flowers,33
music,35
diapers_and_hygiene,32
construction_tools_lights,294


In [0]:
delivery_delay_ml_data.write.format("delta").mode("overwrite")\
    .option("mergeSchema", "false") \
        .saveAsTable("project.gold.delivery_delay")